In [3]:
# !pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as  plt
from matplotlib_venn import venn2
import plotly.express as px
import plotly.graph_objects as go
import ast


# 그래프 해상도 높이기
try:
    %config InlineBackend.figure_format = 'retina'
except Exception as e:
    print(f'💩 {e}')



# 경고 무시
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    os.chdir('/content/drive/MyDrive/파트4')
    print('✅ Succesful access google_drive_directory')
    
except Exception as e:
    print('🤗 Hello vscode')
        
## get_df 함수
def get_df(db_name, table_name):
    if table_name in ['accounts_user', 'accounts_blockrecord']:
        table_name = pd.read_parquet(
            f"gs://high_project/{db_name}/{table_name}.parquet", 
            storage_options={'token' : API_KEY_PATH})
    else:
        table_name = pd.read_csv(
            f"gs://high_project/{db_name}/{table_name}.csv",
            storage_options={'token' : API_KEY_PATH}
            )
    return table_name
    

## literal_eval 형변환 함수
def to_literal_eval(df, column):
    return  df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])


# 리스트 내에 드랍 유저가 있는지 확인하는 함수
def find_drop_users(df, column):
    drop_users = [831956, 1580627, 1580689, 1580626, 995177]
    print(f'{column}:')
    for i in drop_users:
        count_drop_rows = len(df[df[column].apply(lambda x: i in x)])
        if count_drop_rows != 0:
            print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
        else:
            print(f"✅ 관리자 {i} 포함행 없음")
            
            
# 데이트타임형으로 변환 및 기간 전처리
def set_datetime(df, column):
    df[column] =  pd.to_datetime(df[column])
    print(f'✅ {column}데이트 타입 형변환 및 기간 전처리 완료')
    
    

            
API_KEY_PATH ='/home/project_yujin/API_KEY/sprintda03-yujin.json'

🤗 Hello vscode


In [4]:
events = get_df('votes', 'events')
set_datetime(events, 'created_at')
events.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,title,plus_point,event_type,is_expired,created_at
0,1,코드잇 은행 가입 이벤트,500,FCFS,1,2023-06-20 11:56:38
1,2,코드잇 멤버십 가입 이벤트,1000,FCFS,1,2023-08-08 07:43:45
2,3,예고 영상 기대평 이벤트,500,FCFS,1,2023-09-24 17:05:59


In [5]:
event_receipts = get_df('votes', 'event_receipts')
event_receipts = event_receipts[['id' ,'user_id' ,'event_id' ,'plus_point' ,'created_at']]
set_datetime(event_receipts, 'created_at')
event_receipts = event_receipts.drop(81)
event_receipts.head()

✅ created_at데이트 타입 형변환 및 기간 전처리 완료


,id,user_id,event_id,plus_point,created_at
0,2,1193618,1,500,2023-06-22 09:25:16
1,3,928351,1,500,2023-06-22 09:38:53
2,4,904872,1,500,2023-06-22 10:32:15
3,5,974697,1,500,2023-06-22 13:03:06
4,6,1168260,1,500,2023-06-22 13:40:38


In [6]:
# # 날자 파생 컬럼 추가
# event_receipts['C_y_m_d'] = event_receipts['created_at'].dt.to_period('D').astype('str')
# event_receipts['C_y_m'] = event_receipts['created_at'].dt.to_period('M').astype('str')
# event_receipts['C_hour'] = event_receipts['created_at'].dt.hour
# event_receipts['C_weekday'] = event_receipts['created_at'].dt.weekday
# event_receipts['C_is_weekend'] = event_receipts['created_at'].dt.weekday >= 5
# event_receipts['C_time_of_day'] = pd.cut(event_receipts['created_at'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

In [ ]:
# event_id_C_y_m_d = event_receipts.groupby(['event_id'])['C_y_m_d'].value_counts().reset_index().sort_values(by=['event_id', 'C_y_m_d'])

# # 별로 나누기
# event1 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 1]
# event2 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 2]
# event3 = event_id_C_y_m_d[event_id_C_y_m_d['event_id'] == 3]

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=event1['C_y_m_d'],
#     y=event1['count'],
#     mode='lines+markers',
#     name='Weekday',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=event2['C_y_m_d'],
#     y=event2['count'],
#     mode='lines+markers',
#     name='Weekend',
#     line=dict(color='orange')
# ))

# # event3 라인
# fig.add_trace(go.Scatter(
#     x=event3['C_y_m_d'],
#     y=event3['count'],
#     mode='lines+markers',
#     name='Weekend',
#     line=dict(color='purple')
# ))

# fig.show()

In [ ]:
# event_id_C_hour = event_receipts.groupby(['event_id'])['C_hour'].value_counts().reset_index().sort_values(by=['event_id', 'C_hour'])
# event1 = event_id_C_hour.query('event_id == 1')
# event2 = event_id_C_hour.query('event_id == 2')
# event3 = event_id_C_hour.query('event_id == 3')

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=event1['C_hour'],
#     y=event1['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=event2['C_hour'],
#     y=event2['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='orange')
# ))

# # event3 라인ㅁ
# fig.add_trace(go.Scatter(
#     x=event3['C_hour'],
#     y=event3['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='purple')
# ))
# fig.update_layout(
#     title='이벤트별 참가시간대',
#     xaxis_title='Hour of Day',
#     yaxis_title='Count',
#     xaxis=dict(tickmode='linear', dtick=1),
#     template='plotly_white',
#     width=1000,
#     height=500
# )

# fig.show()

In [ ]:
# from datetime import timedelta
# kst = event_receipts[['event_id', 'created_at']]
# kst['kst_created_at'] = pd.to_datetime(kst['created_at'], utc=True) + timedelta(hours=9)

# kst['C_y_m_d'] = kst['kst_created_at'].dt.to_period('D').astype('str')
# kst['C_y_m'] = kst['kst_created_at'].dt.to_period('M').astype('str')
# kst['C_hour'] = kst['kst_created_at'].dt.hour
# kst['C_weekday'] = kst['kst_created_at'].dt.weekday
# kst['C_is_weekend'] = kst['kst_created_at'].dt.weekday >= 5
# kst['C_time_of_day'] = pd.cut(kst['kst_created_at'].dt.hour, bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'], right=False)

In [ ]:
# kst_event_id_C_hour = kst.groupby(['event_id'])['C_hour'].value_counts().reset_index().sort_values(by=['event_id', 'C_hour'])
# kst_event1 = kst_event_id_C_hour.query('event_id == 1')
# kst_event2 = kst_event_id_C_hour.query('event_id == 2')
# kst_event3 = kst_event_id_C_hour.query('event_id == 3')

# fig = go.Figure()

# # event1 라인
# fig.add_trace(go.Scatter(
#     x=kst_event1['C_hour'],
#     y=kst_event1['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='skyblue')
# ))

# # event2 라인
# fig.add_trace(go.Scatter(
#     x=kst_event2['C_hour'],
#     y=kst_event2['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='orange')
# ))

# # event3 라인ㅁ
# fig.add_trace(go.Scatter(
#     x=kst_event3['C_hour'],
#     y=kst_event3['count'],
#     mode='lines+markers',
#     name='',
#     line=dict(color='purple')
# ))
# fig.update_layout(
#     title='이벤트별 참가시간대',
#     xaxis_title='Hour of Day',
#     yaxis_title='Count',
#     xaxis=dict(tickmode='linear', dtick=1),
#     template='plotly_white',
#     width=1000,
#     height=500
# )

# fig.show()

In [7]:
accounts_paymenthistory = get_df('votes', 'accounts_paymenthistory')
accounts_paymenthistory.head()

,id,productId,phone_type,created_at,user_id
0,6,heart.777,A,2023-05-13 21:28:34,1211127
1,7,heart.777,A,2023-05-13 21:29:39,1151343
2,8,heart.777,A,2023-05-13 21:31:33,1002147
3,9,heart.777,A,2023-05-13 21:31:39,1095040
4,11,heart.777,A,2023-05-13 21:34:32,1164081


In [9]:
accounts_failpaymenthistory = get_df('votes', 'accounts_failpaymenthistory')
accounts_failpaymenthistory.head()

,id,productId,phone_type,created_at,user_id
0,6,heart.200,A,2023-05-14 05:49:22,1055891
1,7,heart.777,A,2023-05-14 08:17:21,1152151
2,8,heart.777,A,2023-05-14 10:11:46,986200
3,9,heart.1000,A,2023-05-14 11:53:09,1028261
4,10,heart.777,A,2023-05-14 12:30:47,1235730


In [8]:
accounts_pointhistory = get_df('votes', 'accounts_pointhistory')
accounts_pointhistory = accounts_pointhistory[['id', 'user_id', 'user_question_record_id', 'delta_point', 'created_at']]
accounts_pointhistory = accounts_pointhistory[~accounts_pointhistory.loc[:, [ 'user_id', 'user_question_record_id', 'delta_point', 'created_at']].duplicated()]
accounts_pointhistory = accounts_pointhistory.dropna()
accounts_pointhistory['user_question_record_id'] = accounts_pointhistory['user_question_record_id'].astype(int)
accounts_pointhistory.head()

,id,user_id,user_question_record_id,delta_point,created_at
0,790629,849436,771777,9,2023-04-28 12:27:49
1,790652,849436,771800,9,2023-04-28 12:28:02
2,790664,849436,771812,5,2023-04-28 12:28:09
3,790680,849436,771828,13,2023-04-28 12:28:16
4,790703,849436,771851,5,2023-04-28 12:28:26


In [ ]:
accounts_userquestionrecord = get_df('votes', 'accounts_userquestionrecord')
accounts_userquestionrecord = accounts_userquestionrecord[['id', 'user_id', 'chosen_user_id', 'question_id', 'question_piece_id', \
    'status' ,'answer_status', 'answer_updated_at', 'has_read', 'opened_times', 'report_count', 'created_at']]

accounts_userquestionrecord['status'] = accounts_userquestionrecord['status'].replace({'C':'닫힘'}).replace({'I':'초성열림'}).replace({'B':'차단'})
accounts_userquestionrecord['answer_status'] = accounts_userquestionrecord['answer_status'].replace({'N':'미답변'}).replace({'P':'비공개'}).replace({'A':'공개'})

set_datetime(accounts_userquestionrecord, 'answer_updated_at')
set_datetime(accounts_userquestionrecord, 'created_at')

accounts_userquestionrecord['diff_time'] = accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']
accounts_userquestionrecord['diff_sec_time'] = (accounts_userquestionrecord['answer_updated_at'] - accounts_userquestionrecord['created_at']).dt.total_seconds()

In [ ]:
accounts_userquestionrecord.head(2)

In [ ]:
accounts_userquestionrecord['answer_status'].unique()

In [ ]:
accounts_userquestionrecord['status'].unique()

In [ ]:
accounts_userquestionrecord['status'].unique()

In [ ]:
accounts_userquestionrecord.groupby('answer_status')['diff_sec_time'].mean().reset_index()

In [ ]:
accounts_userquestionrecord.groupby('status')['diff_sec_time'].mean().reset_index()

In [ ]:
accounts_userquestionrecord.groupby(['has_read', 'answer_status'])['status'].value_counts().reset_index()

In [ ]:
accounts_userquestionrecord.groupby('answer_status')['status'].value_counts().reset_index()

In [ ]:
polls_question = get_df('votes', 'polls_question')
polls_question

In [ ]:
polls_questionpiece = get_df('votes', 'polls_questionpiece')
polls_questionpiece

In [ ]:
polls_questionset = get_df('votes', 'polls_questionset')
polls_questionset

In [ ]:
accounts_userquestionrecord.head()

In [ ]:
accounts_userquestionrecord['opened_times'].value_counts()

In [ ]:
accounts_userquestionrecord.groupby(['has_read', 'status']).size().reset_index()

In [ ]:
accounts_userquestionrecord.groupby(['has_read', 'answer_status']).size().reset_index()

In [ ]:
accounts_userquestionrecord.groupby(['status', 'answer_status']).size().reset_index()

In [ ]:
accounts_userquestionrecord.groupby(['status', 'has_read']).size().reset_index()

In [ ]:
polls_question = get_df('votes', 'polls_question')
polls_question['question_text'].value_counts()

In [ ]:
polls_questionpiece = get_df('votes', 'polls_questionpiece')
polls_questionpiece.groupby(['is_skipped', 'is_voted']).size().reset_index()

In [ ]:
polls_questionpiece = get_df('votes', 'polls_questionpiece')
accounts_pointhistory = get_df('votes', 'accounts_pointhistory')

In [ ]:
accounts_pointhistory.head()

In [ ]:
import ast
accounts_attendance['attendance_date_list'] = accounts_attendance['attendance_date_list'].apply(ast.literal_eval)
expanded_df = accounts_attendance.explode('attendance_date_list').reset_index(drop=True)

In [ ]:
polls_questionpiece.head()

In [ ]:
polls_questionset['status'] = polls_questionset['status'].replace({'C' : '닫침'}).replace({'O' : '열림'}).replace({'F' : '종료'})
polls_questionset['question_piece_id_list'] = to_literal_eval(polls_questionset, 'question_piece_id_list')

expanded_polls_questionset = polls_questionset.explode('question_piece_id_list').reset_index(drop=True)
expanded_polls_questionset = expanded_polls_questionset[['id', 'user_id', 'status' ,'created_at', 'opening_time', 'question_piece_id_list']]
expanded_polls_questionset = expanded_polls_questionset.rename(columns={'question_piece_id_list':'question_piece_id'})

In [ ]:
len(expanded_polls_questionset)

In [ ]:
df = pd.merge(expanded_polls_questionset, polls_questionpiece,
    left_on='question_piece_id',
    right_on='id',
    how='inner' , # 또는 'left', 'right', 'outer' 원하는 방식으로
    suffixes=('_questionset', '_questionpiece')  
)

In [ ]:
df =  df[['user_id', 'question_piece_id', 'question_id' ,'is_skipped', 'is_voted' ,'status', 'created_at_questionset', 'opening_time', 'created_at_questionpiece']]

In [ ]:
set_datetime(df, 'created_at_questionset')
set_datetime(df, 'created_at_questionpiece')

In [ ]:
df['diff_created_at'] = (df['created_at_questionset'] - df['created_at_questionpiece']).dt.total_seconds()
df['diff_created_at'].value_counts()

# created_at_questionpiece -> created_at_questionset
df

In [ ]:
accounts_userquestionrecord.head()

In [ ]:
df2 = pd.merge(df, accounts_userquestionrecord,
    left_on='question_piece_id',
    right_on='question_piece_id',
    how='inner' , # 또는 'left', 'right', 'outer' 원하는 방식으로
    suffixes=('_df', '_questionrecord')  
)

In [ ]:
df2 = df2.drop(columns='question_id_questionrecord')

In [ ]:
df2[df2['user_id_df'] != df2['user_id_questionrecord']]

In [ ]:
df2.head()
df2 = df2.rename(columns={'id':'record_id'})

In [ ]:

df3 = df2[['record_id', 'user_id_df' ,'question_piece_id', 'chosen_user_id', 'is_skipped', 'is_voted', 'status_df', 'status_questionrecord' ,'answer_status', 'has_read', 'opened_times' ]]
df3

df3 = df3.rename(columns={'user_id_df':'user_id'})

In [ ]:
accounts_pointhistory = accounts_pointhistory.dropna()

In [ ]:
accounts_pointhistory.head()
accounts_pointhistory = accounts_pointhistory.rename(columns={'user_question_record_id':'question_record_id'})

In [ ]:
polls_questionpiece['id'].nunique()

In [ ]:
df3.head(2)

In [ ]:
accounts_pointhistory.head(2)

In [ ]:
df4 = pd.merge(df3, accounts_pointhistory, 
            left_on='record_id',
            right_on='question_record_id', how='inner')

In [ ]:
df5 = df4[df4['delta_point'] < 0]

In [ ]:
df5['status_questionrecord'].value_counts()

In [ ]:
df5['status_df'].value_counts()

In [ ]:
df5['answer_status'].value_counts()

In [ ]:
df5['has_read'].value_counts()

In [ ]:
df6 = df4[df4['delta_point'] > 0] # 2227344
df6[df6['user_id_x'] == df6['user_id_y']] # 1219513
df6[df6['user_id_x'] != df6['user_id_y']] # 1007831

In [ ]:
df5[df5['user_id_x'] == df5['user_id_y']]

accounts_pointhistory

In [ ]:
result_df = pd.concat([accounts_pointhistory, polls_questionpiece])
result_df = result_df.sort_values(by='created_at')

In [ ]:
result_df.sort_values(by=['created_at'])

In [ ]:
accounts_pointhistory